In [63]:
import os
import pyproj
# os.environ["PROJ_LIB"] = pyproj.datadir.get_data_dir()
import numpy as np
import laspy
import rasterio
from tqdm import tqdm
from rasterio.transform import from_origin
from rasterio.crs import CRS
from PIL import Image, ImageChops
import matplotlib.pyplot as plt

In [ ]:
def extract_MNT_from_PC(src_in_file, src_out_file, resolution=0.5, verbose=False):
    # --------------------------------------------------
    # Load LAS
    # --------------------------------------------------
    las = laspy.read(src_in_file)

    x = las.x
    y = las.y
    z = las.z
    cls = las.classification

    # --------------------------------------------------
    # Keep only ground (2) and buildings (6)
    # --------------------------------------------------
    mask = cls == 2

    x = x[mask]
    y = y[mask]
    z = z[mask]

    # --------------------------------------------------
    # Compute grid extent
    # --------------------------------------------------
    xmin, xmax = x.min(), x.max()
    ymin, ymax = y.min(), y.max()

    width = int(np.ceil((xmax - xmin) / resolution))
    height = int(np.ceil((ymax - ymin) / resolution))

    # --------------------------------------------------
    # Compute pixel indices
    # --------------------------------------------------
    col = ((x - xmin) / resolution).astype(int)
    row = ((ymax - y) / resolution).astype(int)  # flip Y for raster

    # --------------------------------------------------
    # Initialize raster with very low values
    # --------------------------------------------------
    dsm = np.full((height, width), -np.inf, dtype=np.float32)

    # --------------------------------------------------
    # Rasterize using max
    # --------------------------------------------------
    np.maximum.at(dsm, (row, col), np.array(z))

    # Replace empty cells with nodata
    nodata = np.nan
    dsm[dsm == -np.inf] = nodata

    # --------------------------------------------------
    # Write GeoTIFF
    # --------------------------------------------------
    transform = from_origin(xmin, ymax, resolution, resolution)
    
    # crs = las.header.parse_crs()
    # if crs is None:
    #     crs = "EPSG:2056"

    with rasterio.open(
        src_out_file,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype=np.float32,
        # crs=CRS.from_epsg(2056),
        transform=transform,
        nodata=nodata,
    ) as dst:
        dst.write(dsm, 1)

    if verbose:
        print("DSM written to:", src_out_file)

In [37]:
os.environ['PROJ_LIB'] = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\.venv\lib\site-packages\pyproj\proj_dir\share\proj"
os.environ['PROJ_DATA'] = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\.venv\Lib\site-packages\rasterio\proj_data"
# D:\TerraNum_softwares\ToolMap\share\proj
src_pc_files = "data/movement_stcergue/old"
src_out_files = "data/movement_stcergue/MNT"

os.makedirs(src_out_files, exist_ok=True)

list_files = [x for x in os.listdir(src_pc_files) if os.path.splitext(x)[1] in ['.las', '.laz']]
for _, file in tqdm(enumerate(list_files), total=len(list_files)):
    src_in_file = os.path.join(src_pc_files, file)
    src_out_file = os.path.join(src_out_files, file.replace('.las', '_DEM.tif'))
    
    extract_MNT_from_PC(src_in_file, src_out_file, resolution=0.5)

100%|██████████| 9/9 [00:13<00:00,  1.46s/it]


In [107]:
def substract_rasters(src_img1, src_img2, src_out_file):
    img1 = rasterio.open(src_img1)
    img2 = rasterio.open(src_img2)
    img1_arr = img1.read().squeeze(0)
    img2_arr = img2.read().squeeze(0)    

    sub = img1_arr - img2_arr

    with rasterio.open(
        src_out_file,
        "w",
        width=img1_arr.shape[0],
        height=img1_arr.shape[0],
        driver="GTiff",
        count=1,
        dtype=np.float32,
        # crs=CRS.from_epsg(2056),
        transform=img1.transform,
        nodata=np.nan,
    ) as dst:
        dst.write(sub, 1)

In [105]:
old_year = 2018
new_year = 2024
src_dem = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\movement_stcergue\MNT"
src_res = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\movement_stcergue\Difference"
lst_dem_old = [x for x in os.listdir(src_dem) if str(old_year) in x]
lst_dem_new = [x for x in os.listdir(src_dem) if str(new_year) in x]

assert [x.replace(str(old_year), '') for x in lst_dem_old] == [x.replace(str(new_year), '') for x in lst_dem_new]

for id_file, file in tqdm(enumerate(lst_dem_old), total=len(lst_dem_old)):
    substract_rasters(
        os.path.join(src_dem, lst_dem_old[id_file]),
        os.path.join(src_dem, lst_dem_new[id_file]),
        os.path.join(src_res, file.replace(str(old_year), 'diff')),
        )


100%|██████████| 9/9 [00:00<00:00, 14.91it/s]
